# Google Search Ranking & Discoverability Capstone

## Refresh / Content Opportunity Scoring

**Question:** Which content pages should be prioritized for review based on observable search, content-age, and engagement signals?

**Output:** a ranked review queue with reason codes, supported by a Random Forest model and compared with a transparent baseline.

> This project is decision-support. It does not claim to predict Google's ranking algorithm or prove that refreshing a page causes better performance.

## 1. Question

The project supports the decision of **which content pages an editor should review first** when review capacity is limited.

The unit of analysis is one content page in the anonymized starter dataset. The output is a ranked priority score and action recommendation. A human reviewer can then decide whether to refresh, improve, or simply monitor the page.

A false positive can waste editorial time on a page that does not need attention. A false negative can cause a potentially important page to be missed. Machine learning is useful because the decision depends on several observable signals rather than one simple rule.

In [ ]:
import os
import pandas as pd
import numpy as np

REPO_PATH = "/content/Flyrank_ML"

if not os.path.exists(REPO_PATH):
    !git clone https://github.com/DiyaRana7/Flyrank_ML.git

os.chdir(REPO_PATH)

DATA_PATH = "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("Unique pages:", df["content_id"].nunique())
print("Unique clients:", df["client_id"].nunique())
print("Columns:", len(df.columns))

## 2. Data

This capstone uses the **30,000-row anonymized starter dataset** already used in the earlier assignments.

The dataset contains observable content and search-performance fields such as impressions, clicks, sessions, CTR, average position, content age, freshness, and engagement measures.

The target used in the model is `trend_direction == "down"`. This is an **observed current-window proxy**, not a clean future-window outcome. That distinction is important: the results show performance on this defined proxy and should not be described as proof of future decline.

Identifiers such as `content_id` and `client_id` are used only for grouping and reporting checks, never as predictive features. Outcome-derived fields such as `trend_direction` and `trend_pct` are excluded from the feature matrix.

The starter dataset is anonymized and contains no client names, domains, private queries, or raw URLs.

In [ ]:
print("Target distribution:")
target = (df["trend_direction"] == "down").astype(int)
print(target.value_counts())
print("\nDeclining rate:", round(target.mean(), 3))

print("\nMissing values in selected modeling fields:")
candidate = [
    "impressions_90d","clicks_90d","sessions_90d","users_90d",
    "engaged_sessions_90d","days_with_impressions","days_with_sessions",
    "impressions_last_30d","clicks_last_30d","sessions_last_30d",
    "impressions_prev_30d","clicks_prev_30d","sessions_prev_30d",
    "content_age_days","days_since_last_update","ctr","avg_position",
    "engagement_rate","scroll_rate","ai_traffic_pct"
]
display(df[candidate].isnull().sum().sort_values(ascending=False).head(10).to_frame("missing_count"))

## 3. Methodology

### Baseline

The Week-4 baseline assigns points for staleness, visibility, and average position:

- 180+ days since update: +2
- 365+ days since update: +1 additional
- 500+ 90-day impressions: +2
- 5,000+ 90-day impressions: +1 additional
- average position > 10: +1

Score ≥ 4 becomes `REVIEW_REFRESH`; otherwise `MONITOR`.

### Model

A Random Forest classifier is used because it can combine several numeric signals and capture non-linear interactions without requiring a very complex model.

### Features

The final feature set contains 21 observable signals:

`impressions_90d`, `clicks_90d`, `sessions_90d`, `users_90d`, `engaged_sessions_90d`, `days_with_impressions`, `days_with_sessions`, `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`, `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d`, `content_age_days`, `days_since_last_update`, `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`.

`trend_direction`, `trend_pct`, `content_id`, and `client_id` are not used as model features.

### Validation

The evaluation uses a **client-grouped holdout** so that whole clients are kept out of training. This reduces the risk that the model benefits from seeing the same client's patterns in both training and test data.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)

feature_cols = candidate
X = df[feature_cols].copy()
X = X.fillna(X.median(numeric_only=True))

y = (df["trend_direction"] == "down").astype(int)
groups = df["client_id"]

splitter = GroupShuffleSplit(n_splits=1, test_size=0.33, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

train_clients = set(df.iloc[train_idx]["client_id"])
test_clients = set(df.iloc[test_idx]["client_id"])

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Client overlap:", train_clients & test_clients)
print("Training declining rate:", round(y_train.mean(), 3))
print("Test declining rate:", round(y_test.mean(), 3))

In [ ]:
model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)
model.fit(X_train, y_train)

model_pred = model.predict(X_test)
model_prob = model.predict_proba(X_test)[:, 1]

def precision_at_k(y_true, scores, k=50):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)
    top_idx = np.argsort(-scores)[:k]
    return y_true[top_idx].mean()

model_metrics = {
    "accuracy": accuracy_score(y_test, model_pred),
    "precision": precision_score(y_test, model_pred, zero_division=0),
    "recall": recall_score(y_test, model_pred, zero_division=0),
    "f1": f1_score(y_test, model_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test, model_prob),
    "average_precision": average_precision_score(y_test, model_prob),
    "precision_at_50": precision_at_k(y_test, model_prob, 50)
}

print("Random Forest metrics:")
for k, v in model_metrics.items():
    print(f"{k}: {v:.3f}")

In [ ]:
# Recreate the Week-4 baseline on exactly the same test rows
baseline_test = df.iloc[test_idx].copy()
baseline_test["score"] = 0

baseline_test["score"] += (baseline_test["days_since_last_update"] >= 180).astype(int) * 2
baseline_test["score"] += (baseline_test["days_since_last_update"] >= 365).astype(int)
baseline_test["score"] += (baseline_test["impressions_90d"] >= 500).astype(int) * 2
baseline_test["score"] += (baseline_test["impressions_90d"] >= 5000).astype(int)
baseline_test["score"] += (baseline_test["avg_position"] > 10).astype(int)

baseline_pred = (baseline_test["score"] >= 4).astype(int)

baseline_metrics = {
    "accuracy": accuracy_score(y_test, baseline_pred),
    "precision": precision_score(y_test, baseline_pred, zero_division=0),
    "recall": recall_score(y_test, baseline_pred, zero_division=0),
    "f1": f1_score(y_test, baseline_pred, zero_division=0),
    "precision_at_50": precision_at_k(y_test, baseline_test["score"], 50)
}

comparison = pd.DataFrame({
    "metric": ["accuracy", "precision", "recall", "f1", "precision_at_50"],
    "Week-4 baseline": [baseline_metrics[m] for m in ["accuracy","precision","recall","f1","precision_at_50"]],
    "Random Forest": [model_metrics[m] for m in ["accuracy","precision","recall","f1","precision_at_50"]]
})

display(comparison.round(3))
print("\nTest-set declining base rate:", round(y_test.mean(), 3))

## 4. Results and interpretation

On the evaluated client-grouped holdout, the Random Forest showed substantially stronger measured performance than the Week-4 rule on the selected declining-trend proxy.

The most decision-relevant metric is Precision@50 because the practical use case is a limited review queue. The base rate is reported alongside it so the ranking result is not interpreted without context.

Feature importance is descriptive: an important feature is a signal the model used, not proof that the feature causes decline.

In [ ]:
importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

display(importance.head(10))

cm = confusion_matrix(y_test, model_pred)
print("Confusion matrix:")
print(cm)

tn, fp, fn, tp = cm.ravel()
print("False positives:", fp)
print("False negatives:", fn)

## 5. Limitations

- The target is `trend_direction == "down"`, an observed current-window proxy rather than a future-window outcome.
- Therefore, this work should not be presented as proof that the model predicts future search performance.
- A client-grouped split reduces client memorization risk, but it does not establish performance across all future time periods.
- Feature importance shows model reliance, not causality.
- A high-priority page may already be performing well, may have a seasonal explanation, or may not benefit from a refresh.
- The data cannot prove what causes Google rankings to change.
- The work does not show that a content refresh causes recovery or increased traffic.
- Final actions require human editorial review.

## 6. Ranked recommendations

The action playbook uses the transparent baseline to create an interpretable review queue.

Recommended workflow:

1. Start with the highest-ranked pages.
2. Read the reason code and supporting signals.
3. Check whether the page is actually outdated, incomplete, inaccurate, or otherwise worth attention.
4. Rule out seasonality, consolidation, low-volume noise, or other explanations where possible.
5. Choose an action: refresh, improve, monitor, or no action.
6. Do not automatically publish, delete, merge, or rewrite content from the score alone.

In [ ]:
# Recreate the full ranked baseline queue for the paper
queue = df.copy()
queue["score"] = 0
queue["score"] += (queue["days_since_last_update"] >= 180).astype(int) * 2
queue["score"] += (queue["days_since_last_update"] >= 365).astype(int)
queue["score"] += (queue["impressions_90d"] >= 500).astype(int) * 2
queue["score"] += (queue["impressions_90d"] >= 5000).astype(int)
queue["score"] += (queue["avg_position"] > 10).astype(int)

queue["reason_code"] = np.select(
    [
        (queue["days_since_last_update"] >= 180) & (queue["impressions_90d"] >= 500),
        queue["days_since_last_update"] >= 180,
        queue["impressions_90d"] >= 500
    ],
    ["STALE_VISIBLE", "STALE", "VISIBLE"],
    default="OTHER"
)

queue["action"] = np.where(queue["score"] >= 4, "REVIEW_REFRESH", "MONITOR")
queue["confidence_note"] = np.select(
    [queue["score"] >= 6, queue["score"] >= 4],
    ["Multiple strong priority signals", "Meets baseline review threshold"],
    default="Lower priority based on baseline"
)

queue = queue.sort_values(["score", "impressions_90d"], ascending=[False, False]).reset_index(drop=True)
queue["rank"] = np.arange(1, len(queue) + 1)

print("Total pages:", len(queue))
print("\nAction distribution:")
display(queue["action"].value_counts().to_frame("count"))

print("\nReason-code distribution:")
display(queue["reason_code"].value_counts().to_frame("count"))

print("\nTop 20:")
display(queue[["rank","score","reason_code","action","confidence_note",
               "days_since_last_update","impressions_90d","avg_position","ctr"]].head(20))

## 7. Artifacts the paper embeds

The following outputs are generated from the same notebook so the paper can be traced back to the analysis:

- model-vs-baseline metrics
- top feature importance table
- confusion matrix
- ranked action queue
- action and reason-code distributions

Only public-safe aggregated results and pseudonymized identifiers should be shown publicly. Do not publish raw client names, domains, URLs, private queries, credentials, or raw exports.

In [ ]:
import os
import matplotlib.pyplot as plt

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

# Metrics receipt
metrics_receipt = {
    "test_rows": int(len(test_idx)),
    "test_declining_base_rate": float(y_test.mean()),
    "baseline_precision_at_50": float(baseline_metrics["precision_at_50"]),
    "model_precision_at_50": float(model_metrics["precision_at_50"]),
    "baseline_f1": float(baseline_metrics["f1"]),
    "model_f1": float(model_metrics["f1"]),
    "model_roc_auc": float(model_metrics["roc_auc"]),
    "model_average_precision": float(model_metrics["average_precision"]),
    "random_seed": 42,
    "validation": "client_grouped_holdout"
}

import json
with open("work/outputs/capstone_metrics.json", "w") as f:
    json.dump(metrics_receipt, f, indent=2)

# Public-safe recommendation summary
summary = (
    queue.groupby(["action", "reason_code"], dropna=False)
    .size()
    .reset_index(name="pages")
)
summary.to_csv("work/outputs/capstone_action_summary.csv", index=False)

# Feature importance figure
top_imp = importance.head(10).sort_values("importance")
plt.figure(figsize=(8, 5))
plt.barh(top_imp["feature"], top_imp["importance"])
plt.xlabel("Random Forest feature importance")
plt.title("Top 10 Model Features")
plt.tight_layout()
plt.savefig("work/figures/capstone_feature_importance.png", dpi=160)
plt.show()

# Baseline/model comparison figure
plot_df = comparison[comparison["metric"].isin(["precision_at_50","f1","roc_auc"])].copy()
plot_df = plot_df.melt(id_vars="metric", var_name="method", value_name="value")
plt.figure(figsize=(8, 5))
for method in plot_df["method"].unique():
    sub = plot_df[plot_df["method"] == method]
    plt.plot(sub["metric"], sub["value"], marker="o", label=method)
plt.ylabel("Score")
plt.title("Model vs Baseline on the Same Test Split")
plt.legend()
plt.tight_layout()
plt.savefig("work/figures/capstone_model_vs_baseline.png", dpi=160)
plt.show()

print("Artifacts written.")
print("Metrics:", "work/outputs/capstone_metrics.json")
print("Action summary:", "work/outputs/capstone_action_summary.csv")
print("Figures:", "work/figures/")

## Self-check

- [x] Research question and decision are explicit.
- [x] Data source and exclusions are documented.
- [x] The target is described honestly as an observed current-window proxy.
- [x] Baseline and Random Forest use the same evaluation rows.
- [x] Client-grouped validation is checked for zero overlap.
- [x] Leakage-prone target/identifier fields are excluded from features.
- [x] Base rate and decision-relevant ranking metrics are reported.
- [x] Errors and feature importance are inspected.
- [x] Recommendations are human-review decision support.
- [x] Public-safe artifacts are generated under `work/outputs/` and `work/figures/`.
- [x] No causal claims or claims about Google's algorithm are made.